# Pretty Formula One

This notebook is used to fetch data from certain seasons and save them into files to be uploaded into hosted database

In [25]:
YEAR = 2021

## Drivers

In [26]:
import fastf1
import json
from pprint import pprint
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

session = fastf1.get_session(YEAR, 'Monaco', 'R')
session.load(laps=False, telemetry=False, weather=False)
drivers_list = []
seen_drivers = set()
for _, row in session.results.iterrows():
    driver_slug = f"{row['DriverId']}_{YEAR}"
    if driver_slug not in seen_drivers:
        drivers_list.append({
            "id": driver_slug,
            "team": row["TeamName"],
            "abbreviation": row["Abbreviation"],
            "name": row["FullName"],
            "teamLogo": f"/assets/icons/{row['TeamName'].lower().replace(' ', '-')}.png",
        })
        seen_drivers.add(driver_slug)
        
os.makedirs(f'data/{YEAR}', exist_ok=True)
with open(f"./data/{YEAR}/drivers_{YEAR}.json", "w") as f:
    json.dump(drivers_list, f, separators=(',', ':'))

## Race Results

In [27]:
import fastf1
import json
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

os.makedirs(f'data/{YEAR}', exist_ok=True)
with open(f"./data/{YEAR}/drivers_{YEAR}.json", "r") as f:
    drivers_list = json.load(f)
    
schedule = fastf1.get_event_schedule(YEAR)
races = schedule["RoundNumber"][schedule["RoundNumber"] > 0].tolist()
    
all_rounds_data = []
for ROUND_ID in races:

    race_session = fastf1.get_session(YEAR, ROUND_ID, "R")
    race_session.load(telemetry=False, weather=False)

    race_points_map = {
        f"{row['DriverId']}_{YEAR}": int(row['Points']) 
        for _, row in race_session.results.iterrows()
    }

    id_to_number_map = {
        f"{row['DriverId']}_{YEAR}": row['DriverNumber'] 
        for _, row in race_session.results.iterrows()
    }

    sprint_points_map = {}
    has_sprint = False
    try:
        sprint_session = fastf1.get_session(YEAR, ROUND_ID, "S")
        sprint_session.load(laps=False, telemetry=False, weather=False)
        has_sprint = True
        sprint_points_map = {
            row['DriverNumber']: int(row['Points']) 
            for _, row in sprint_session.results.iterrows()
        }
    except Exception:
        pass

    race_results_list = []
    for driver in drivers_list:
        d_id = driver["id"]
        r_points = race_points_map.get(d_id, 0)
        s_points = 0
        if has_sprint:
            d_number = id_to_number_map.get(d_id)
            if d_number:
                s_points = sprint_points_map.get(d_number, 0)
                
        laps = race_session.laps
        tyre_data = laps[['Compound', 'LapNumber', 'DriverNumber', 'Stint']]
        lap_tyre_data = {}
        for _, row in tyre_data.iterrows():
            if row["Compound"] == "nan":
                continue
            driver_number = row["DriverNumber"]
            if driver_number not in lap_tyre_data:
                lap_tyre_data[driver_number] = []
            last_compound = lap_tyre_data[driver_number][-1]["compound"] if lap_tyre_data[driver_number] else None
            if row["Compound"] != last_compound:
                lap_tyre_data[driver_number].append({
                    "compound": row["Compound"],
                    "lapStart": row["LapNumber"],
                    "lapEnd": row["LapNumber"],
                    "stint": row["Stint"]
                })
            else:
                lap_tyre_data[driver_number][-1]["lapEnd"] = row["LapNumber"]

        race_results_list.append({
            "driver_id": d_id,
            "racePoints": r_points,
            "sprintPoints": s_points,
            "tyre_strat": lap_tyre_data.get(id_to_number_map.get(d_id), [])
        })

    event_info = race_session.event
    country = event_info["Country"]
    schedule = fastf1.get_event_schedule(YEAR)
    total_rounds = len(schedule[schedule["RoundNumber"] > 0])

    round_data = {
        "id": int(event_info["RoundNumber"]),
        "year": YEAR,
        "index": int(event_info["RoundNumber"]),
        "totalRounds": total_rounds,
        "name": event_info["EventName"],
        "nameVerbose": event_info["OfficialEventName"],
        "country": country,
        "backgroundImage": f"/assets/circuits/{country.lower().replace(' ', '_')}.png",
        "results": sorted(race_results_list, key=lambda x: x["racePoints"], reverse=True),
    }
    
    print(f"Processed round {ROUND_ID}: {event_info['EventName']}")

    all_rounds_data.append(round_data)
    
filename = f"./data/{YEAR}/rounds_{YEAR}.json"

os.makedirs(f'data/{YEAR}', exist_ok=True)
with open(filename, "w") as f:
    json.dump(all_rounds_data, f, separators=(',', ':'))

Processed round 1: Bahrain Grand Prix
Processed round 2: Emilia Romagna Grand Prix
Processed round 3: Portuguese Grand Prix
Processed round 4: Spanish Grand Prix
Processed round 5: Monaco Grand Prix
Processed round 6: Azerbaijan Grand Prix
Processed round 7: French Grand Prix
Processed round 8: Styrian Grand Prix
Processed round 9: Austrian Grand Prix
Processed round 10: British Grand Prix
Processed round 11: Hungarian Grand Prix
Processed round 12: Belgian Grand Prix
Processed round 13: Dutch Grand Prix
Processed round 14: Italian Grand Prix
Processed round 15: Russian Grand Prix
Processed round 16: Turkish Grand Prix
Processed round 17: United States Grand Prix
Processed round 18: Mexico City Grand Prix
Processed round 19: São Paulo Grand Prix
Processed round 20: Qatar Grand Prix
Processed round 21: Saudi Arabian Grand Prix
Processed round 22: Abu Dhabi Grand Prix


## All Event Names and Weekend Dates

In [28]:
# getting all the name of the grand prix on the season
import fastf1
import json
from pprint import pprint
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

schedule = fastf1.get_event_schedule(YEAR)
events_dates = []
for event_name in schedule["EventName"].to_list():
    date = schedule[schedule["EventName"] == event_name]["Session1Date"].values[0]
    date = date.strftime("%Y-%m-%dT%H:%M:%S%z")
    events_dates.append({
        "name": event_name.replace("Grand Prix", "").strip(),
        "date": date,
        "season": YEAR,
    })
pprint(events_dates)
os.makedirs(f'data/{YEAR}', exist_ok=True)
with open(f"./data/{YEAR}/dates_{YEAR}.json", "w") as f:
    json.dump(events_dates, f, separators=(',', ':'))

[{'date': '2021-03-12T10:00:00+0300',
  'name': 'Pre-Season Test',
  'season': 2021},
 {'date': '2021-03-26T14:30:00+0300', 'name': 'Bahrain', 'season': 2021},
 {'date': '2021-04-16T11:00:00+0200', 'name': 'Emilia Romagna', 'season': 2021},
 {'date': '2021-04-30T11:30:00+0100', 'name': 'Portuguese', 'season': 2021},
 {'date': '2021-05-07T11:30:00+0200', 'name': 'Spanish', 'season': 2021},
 {'date': '2021-05-20T11:30:00+0200', 'name': 'Monaco', 'season': 2021},
 {'date': '2021-06-04T12:30:00+0400', 'name': 'Azerbaijan', 'season': 2021},
 {'date': '2021-06-18T11:30:00+0200', 'name': 'French', 'season': 2021},
 {'date': '2021-06-25T11:30:00+0200', 'name': 'Styrian', 'season': 2021},
 {'date': '2021-07-02T11:30:00+0200', 'name': 'Austrian', 'season': 2021},
 {'date': '2021-07-16T14:30:00+0100', 'name': 'British', 'season': 2021},
 {'date': '2021-07-30T11:30:00+0200', 'name': 'Hungarian', 'season': 2021},
 {'date': '2021-08-27T11:30:00+0200', 'name': 'Belgian', 'season': 2021},
 {'date': '2

## Race Telemetry

In [29]:
from fastf1.ergast import Ergast
import fastf1
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

ergast = Ergast()
DRIVERS = ergast.get_driver_info(YEAR)["driverId"].to_list()

schedule = fastf1.get_event_schedule(YEAR)
races = schedule["RoundNumber"][schedule["RoundNumber"] > 0].tolist()

for race in races:
    for driver in DRIVERS:
        session = fastf1.get_session(YEAR, race, "R")
        session.load(laps=True, telemetry=True, weather=False)
        driver_slug = f"{driver}_{YEAR}"
        for _, row in session.results.iterrows():
            if row["DriverId"] == driver:
                driver_abv = row["Abbreviation"]
                break
        driver_laps = session.laps.pick_drivers(driver_abv)
        if driver_laps.empty:
            continue
        fastest_lap = driver_laps.pick_fastest()
        if fastest_lap is None:
            fastest_lap = driver_laps.iloc[0]
        start_td = fastest_lap["Time"] - fastest_lap["LapTime"]
        end_td = fastest_lap["Time"]
        telemetry = fastest_lap.get_telemetry()
        telemetry['RelativeSeconds'] = (telemetry['SessionTime'] - start_td).dt.total_seconds()
        telemetry['Compound'] = fastest_lap['Compound']

        position_points = []
        for row in telemetry.itertuples():
            position_points.append((
                round(row.RelativeSeconds,1), 
                round(row.X,1), 
                round(row.Y,1),
                round(row.Z,1),
                round(row.RPM,1),
                round(row.Speed,1), 
                row.nGear, 
                round(row.Throttle,1), 
                row.Brake, 
                row.DRS, 
                row.Status,
                row.Compound
            ))

        os.makedirs(f'data/{YEAR}', exist_ok=True)
        filename = f'data/{YEAR}/telemetry_{driver}_{YEAR}_{race}.csv'

        with open(filename, 'w') as f:
            f.write("seconds,x,y,z,rpm,speed,gear,throttle,brake,drs,status,compound\n")
            for point in position_points:
                f.write(','.join(map(str, point)) + '\n')
                
        d_nam = driver
        d_idx = DRIVERS.index(driver)+1
        d_tot = len(DRIVERS)
        r_idx = races.index(race)+1
        r_tot = len(races)
        t_idx = d_idx*r_idx
        t_tot = d_tot*r_tot
        print(f"Exported points for {d_nam} ({d_idx}/{d_tot}) in race {r_idx} ({r_idx}/{r_tot}) total iterations: [{t_idx}/{t_tot}]")

Exported points for alonso (1/21) in race 1 (1/22) total iterations: [1/462]
Exported points for bottas (2/21) in race 1 (1/22) total iterations: [2/462]
Exported points for gasly (3/21) in race 1 (1/22) total iterations: [3/462]
Exported points for giovinazzi (4/21) in race 1 (1/22) total iterations: [4/462]
Exported points for hamilton (5/21) in race 1 (1/22) total iterations: [5/462]
Exported points for kubica (6/21) in race 1 (1/22) total iterations: [6/462]
Exported points for latifi (7/21) in race 1 (1/22) total iterations: [7/462]
Exported points for leclerc (8/21) in race 1 (1/22) total iterations: [8/462]
Exported points for mazepin (9/21) in race 1 (1/22) total iterations: [9/462]
Exported points for norris (10/21) in race 1 (1/22) total iterations: [10/462]
Exported points for ocon (11/21) in race 1 (1/22) total iterations: [11/462]
Exported points for perez (12/21) in race 1 (1/22) total iterations: [12/462]
Exported points for ricciardo (13/21) in race 1 (1/22) total itera